# Denoising Diffusion Probabilistic Models (DDPM) — learn to undo noise

> Tutorial pair for [`ddpm.py`](ddpm.py).

## 1. Intuition
Take a data point and slowly stir in Gaussian noise until, after many steps, it
is indistinguishable from static. That destruction is *easy* and needs no
learning. The hard, useful direction is the reverse: a network learns to remove
a little noise at a time. Sampling then means starting from pure noise and
running the learned denoiser backwards until a clean sample emerges.

## 2. Concept (the slide)
- **Forward process** $q$: fixed, gradually adds Gaussian noise over $T$ steps.
- **Reverse process** $p_\theta$: learned, removes noise step by step.
- Each $q(x_t\mid x_{t-1})=\mathcal N(\sqrt{1-\beta_t}\,x_{t-1},\beta_t I)$.
- A magic identity gives $q(x_t\mid x_0)$ in **closed form**, so we can jump to
  any noise level in one shot during training.
- The variational bound simplifies to predicting the **noise** $\epsilon$ with a
  plain MSE loss.

## 3. Math derivation — from the VLB to a simple MSE

**Forward marginal in closed form.** Let $\alpha_t=1-\beta_t$ and
$\bar\alpha_t=\prod_{s\le t}\alpha_s$. Composing Gaussians,
$$\boxed{\,q(x_t\mid x_0)=\mathcal N\!\big(\sqrt{\bar\alpha_t}\,x_0,\;(1-\bar\alpha_t)I\big)\,}
\quad\Longleftrightarrow\quad
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\ \epsilon\sim\mathcal N(0,I).$$
This is the same reparameterization trick as the VAE: noise is an external input.

**Tractable posterior.** Bayes on the Gaussian chain gives the reverse
*conditional* in closed form too:
$$q(x_{t-1}\mid x_t,x_0)=\mathcal N\big(\tilde\mu_t(x_t,x_0),\tilde\beta_t I\big),
\quad
\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t .$$

**Variational bound.** Maximizing $\log p_\theta(x_0)$ uses the ELBO
$$\mathbb E_q\Big[\underbrace{\mathrm{KL}(q(x_T|x_0)\,\|\,p(x_T))}_{L_T}
 +\sum_{t>1}\underbrace{\mathrm{KL}(q(x_{t-1}|x_t,x_0)\,\|\,p_\theta(x_{t-1}|x_t))}_{L_{t-1}}
 -\underbrace{\log p_\theta(x_0|x_1)}_{L_0}\Big].$$
Each $L_{t-1}$ is a KL between two Gaussians, i.e. a squared difference of means.

**The simplification.** Parameterize the reverse mean through a noise predictor
$\epsilon_\theta(x_t,t)$:
$$\mu_\theta(x_t,t)=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\Big).$$
Substituting $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$ into the
$L_{t-1}$ means and dropping the $t$-dependent weights yields the famous
**simplified objective**:
$$\boxed{\,L_{\text{simple}}=\mathbb E_{x_0,\,t,\,\epsilon}\big\|\epsilon-\epsilon_\theta(x_t,t)\big\|^2\,}.$$
Just regress the network onto the noise you added.

**Reverse sampler (ancestral).** Sample $x_T\sim\mathcal N(0,I)$ and iterate
$$x_{t-1}=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\Big)+\sqrt{\beta_t}\,z,
\quad z\sim\mathcal N(0,I)\ (z=0\text{ at }t=0).$$

## 4. Model — sinusoidal time embedding + eps-prediction MLP

In [ ]:
# ===== actual implementation from ddpm.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _forward_sample_numpy(x0, abar_t, rng):
    """Sample x_t directly from x_0 using the closed-form marginal."""
    eps = rng.normal(size=x0.shape)
    return np.sqrt(abar_t) * x0 + np.sqrt(1.0 - abar_t) * eps, eps

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    """Linear beta schedule and the precomputed alpha / alpha-bar tensors."""
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    abars = torch.cumprod(alphas, dim=0)
    return betas, alphas, abars

def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """Transformer-style timestep embedding for integer steps t (shape (N,))."""
    half = dim // 2
    freqs = torch.exp(-np.log(10000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1)  # tiny model: 1 thread avoids CPU thrashing
    from sklearn.datasets import make_moons
    X, _ = make_moons(2000, noise=0.05, random_state=SEED)
    X = (X - X.mean(0)) / X.std(0)            # standardize
    X = X.astype(np.float32)

    m = DDPM(data_dim=2, T=200).fit(X, epochs=400)
    print(f"DDPM final eps-MSE = {m.history[-1]:.4f}")

    s = m.sample(2000)
    print(f"  data  mean={X.mean(0).round(2)}  std={X.std(0).round(2)}")
    print(f"  samp  mean={s.mean(0).round(2)}  std={s.std(0).round(2)}")


class EpsMLP(nn.Module):
    """Predict the noise eps added to x_t, conditioned on the timestep t."""

    def __init__(self, data_dim: int = 2, hidden: int = 128, t_dim: int = 32):
        super().__init__()
        self.t_dim = t_dim
        self.net = nn.Sequential(
            nn.Linear(data_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, data_dim))

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        emb = sinusoidal_embedding(t, self.t_dim)
        return self.net(torch.cat([x, emb], dim=1))


class DDPM(nn.Module):
    r"""
    Discrete-time DDPM. Trains the simplified objective

        L = E_{x0, t, eps} || eps - eps_theta(x_t, t) ||^2 ,
        x_t = sqrt(abar_t) x0 + sqrt(1 - abar_t) eps.

    Sampling is the ancestral reverse chain
        x_{t-1} = 1/sqrt(alpha_t) ( x_t - (beta_t / sqrt(1-abar_t)) eps_theta )
                  + sqrt(beta_t) z,   z ~ N(0, I)  (z = 0 at t = 0).
    """

    def __init__(self, data_dim: int = 2, T: int = 200, hidden: int = 128):
        super().__init__()
        self.T = T
        self.model = EpsMLP(data_dim, hidden)
        betas, alphas, abars = make_beta_schedule(T)
        # store schedule as buffers so .to(device) moves them too
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("abars", abars)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, eps: torch.Tensor):
        """Closed-form forward marginal x_t ~ q(x_t | x0)."""
        ab = self.abars[t].unsqueeze(1)
        return ab.sqrt() * x0 + (1.0 - ab).sqrt() * eps

    def loss(self, x0: torch.Tensor) -> torch.Tensor:
        n = len(x0)
        t = torch.randint(0, self.T, (n,), device=x0.device)
        eps = torch.randn_like(x0)
        x_t = self.q_sample(x0, t, eps)
        eps_hat = self.model(x_t, t)
        return F.mse_loss(eps_hat, eps)

    def fit(self, X, epochs: int = 400, batch: int = 256, lr: float = 2e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                loss = self.loss(X[perm[s:s + batch]])
                opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int, data_dim: int = 2):
        """Ancestral sampling: start from noise, denoise down to x_0."""
        dev = next(self.parameters()).device
        x = torch.randn(n, data_dim, device=dev)
        for ti in reversed(range(self.T)):
            t = torch.full((n,), ti, device=dev, dtype=torch.long)
            eps_hat = self.model(x, t)
            alpha = self.alphas[ti]
            abar = self.abars[ti]
            beta = self.betas[ti]
            mean = (x - beta / (1.0 - abar).sqrt() * eps_hat) / alpha.sqrt()
            if ti > 0:
                x = mean + beta.sqrt() * torch.randn_like(x)
            else:
                x = mean
        return x.cpu().numpy()

## 5. Training / sampling — closed-form q_sample, MSE loss, ancestral sampler

In [ ]:
# ===== actual implementation from ddpm.py =====
class DDPM(nn.Module):
    r"""
    Discrete-time DDPM. Trains the simplified objective

        L = E_{x0, t, eps} || eps - eps_theta(x_t, t) ||^2 ,
        x_t = sqrt(abar_t) x0 + sqrt(1 - abar_t) eps.

    Sampling is the ancestral reverse chain
        x_{t-1} = 1/sqrt(alpha_t) ( x_t - (beta_t / sqrt(1-abar_t)) eps_theta )
                  + sqrt(beta_t) z,   z ~ N(0, I)  (z = 0 at t = 0).
    """

    def __init__(self, data_dim: int = 2, T: int = 200, hidden: int = 128):
        super().__init__()
        self.T = T
        self.model = EpsMLP(data_dim, hidden)
        betas, alphas, abars = make_beta_schedule(T)
        # store schedule as buffers so .to(device) moves them too
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("abars", abars)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, eps: torch.Tensor):
        """Closed-form forward marginal x_t ~ q(x_t | x0)."""
        ab = self.abars[t].unsqueeze(1)
        return ab.sqrt() * x0 + (1.0 - ab).sqrt() * eps

    def loss(self, x0: torch.Tensor) -> torch.Tensor:
        n = len(x0)
        t = torch.randint(0, self.T, (n,), device=x0.device)
        eps = torch.randn_like(x0)
        x_t = self.q_sample(x0, t, eps)
        eps_hat = self.model(x_t, t)
        return F.mse_loss(eps_hat, eps)

    def fit(self, X, epochs: int = 400, batch: int = 256, lr: float = 2e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                loss = self.loss(X[perm[s:s + batch]])
                opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int, data_dim: int = 2):
        """Ancestral sampling: start from noise, denoise down to x_0."""
        dev = next(self.parameters()).device
        x = torch.randn(n, data_dim, device=dev)
        for ti in reversed(range(self.T)):
            t = torch.full((n,), ti, device=dev, dtype=torch.long)
            eps_hat = self.model(x, t)
            alpha = self.alphas[ti]
            abar = self.abars[ti]
            beta = self.betas[ti]
            mean = (x - beta / (1.0 - abar).sqrt() * eps_hat) / alpha.sqrt()
            if ti > 0:
                x = mean + beta.sqrt() * torch.randn_like(x)
            else:
                x = mean
        return x.cpu().numpy()

## 6. Train & sample on 2-D two-moons

In [ ]:
demo()

## 7. Visualization — the forward noising process and learned samples

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt, torch
from sklearn.datasets import make_moons
import ddpm as M

X, _ = make_moons(2000, noise=0.05, random_state=0)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")
m = M.DDPM(data_dim=2, T=200).fit(X, epochs=400)

# forward process: x_0 -> x_t for increasing t (closed form)
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
xt0 = torch.tensor(X)
for ax, t in zip(axes, [0, 25, 75, 150, 199]):
    tt = torch.full((len(X),), t, dtype=torch.long)
    eps = torch.randn_like(xt0)
    xt = m.q_sample(xt0, tt, eps).numpy()
    ax.scatter(xt[:, 0], xt[:, 1], s=4, alpha=.3)
    ax.set_title(f"q(x_t|x_0), t={t}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

# reverse: generated samples vs data
s = m.sample(2000)
plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], s=5, alpha=.3, label="data")
plt.scatter(s[:, 0], s[:, 1], s=5, alpha=.4, color="r", label="DDPM samples")
plt.legend(); plt.title("Reverse process samples vs data"); plt.axis("equal")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- The closed-form $q(x_t\mid x_0)$ is what makes training cheap: pick a random
  $t$, noise the sample in one step, regress the noise.
- The simplified MSE loss is an (unweighted) variational bound — it just works.
- Many steps $T$ make *training* easy but *sampling* slow (one network call per
  step). **DDIM** (next file) keeps the same trained network but samples in far
  fewer, deterministic steps.
- Pitfalls: the data should be standardized; too few steps or a bad $\beta$
  schedule leaves residual noise in samples.